# Análisis de texto y ML con PySpark: reseñas de Steam

**Pregunta:** ¿se puede predecir si un jugador *recomienda* un juego solo con el texto de su reseña — y qué palabras separan una reseña positiva de una negativa?

Dataset: `steam_reviews.csv` (~7.8 GB, columnas `app_id`, `app_name`, `review`, `recommended`, `author.*`).

**Cluster:** el de la sección B del README de MLOps (Dataproc `2.1-ubuntu20`, master `e2-highmem-2`, 3 workers `e2-standard-2`, Jupyter con Component Gateway). Es un cluster **chico** (6 vCPU y ~24 GB de RAM entre los workers), así que el notebook trabaja sobre una muestra (`FRACCION`) y convierte el CSV a Parquet una sola vez. Con el cluster B.1 (workers `n1-standard-4`) puedes subir la fracción.

Es el mismo `Pipeline` de `PySpark_Intro` (primer modelo de tweets) y de `Fraud_Detection`, pero con texto: `Tokenizer → StopWords → CountVectorizer → IDF → LogisticRegression`. Aquí `CountVectorizer` reemplaza al `HashingTF` de Intro porque conserva el vocabulario y así podemos ver **qué palabras** pesan en la predicción.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.storagelevel import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

In [ ]:
spark = SparkSession.builder \
    .appName("NLP_Steam_Reviews") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

In [ ]:
# --- Ajusta estas rutas a tu bucket ---
BUCKET = "big-data-lunes-20260223"
RUTA_CSV     = f"gs://{BUCKET}/steam/steam_reviews.csv"
RUTA_PARQUET = f"gs://{BUCKET}/steam/steam_reviews_parquet"   # se crea en la sección 1
RUTA_MODELO  = f"gs://{BUCKET}/modelos/steam_nlp"

# Con el cluster B empieza en 0.05; sube a 0.2, 0.5, 1.0 conforme veas los tiempos.
FRACCION = 0.05
# True la primera vez (lee el CSV completo y lo guarda como Parquet). False después.
CONVERTIR_A_PARQUET = True

## 1. Ingesta y calidad del dato

Las reseñas son texto libre y muchas traen **saltos de línea dentro de comillas**. Igual que en `PySpark_Recommenders` (reviews de anime), hay que leer con `multiLine=True`; si no, Spark parte esas reseñas en varias filas rotas.

Dos costos que conviene tener presentes:
- Un CSV `multiLine` **no se puede dividir** entre executors: un solo task lee los 7.8 GB. Por eso se hace **una vez** y se guarda como Parquet, que sí es paralelo y mucho más rápido de releer.
- No usamos `inferSchema`: sería una pasada extra sobre los 7.8 GB. Todo entra como texto y se convierte después.

In [ ]:
%%time
if CONVERTIR_A_PARQUET:
    raw_csv = spark.read.csv(
        RUTA_CSV, header=True, multiLine=True, escape='"', quote='"'
    )
    # "author.playtime_forever" -> "author_playtime_forever" (los puntos rompen el SQL)
    raw_csv = raw_csv.toDF(*[c.replace(".", "_") for c in raw_csv.columns])
    raw_csv.write.mode("overwrite").parquet(RUTA_PARQUET)

raw = spark.read.parquet(RUTA_PARQUET)
if FRACCION < 1.0:
    raw = raw.sample(fraction=FRACCION, seed=42)
raw.printSchema()

In [ ]:
cols = set(raw.columns)
faltan = {"review", "recommended"} - cols
assert not faltan, f"Faltan columnas obligatorias {faltan}. Encontradas: {sorted(cols)}"

n_raw = raw.count()
label_txt = F.lower(F.col("recommended").cast("string"))
df = (raw
      .withColumn("label", F.when(label_txt.isin("true", "1"), 1.0)
                            .when(label_txt.isin("false", "0"), 0.0))   # cualquier otra cosa -> null
      .filter(F.col("label").isNotNull() & F.col("review").isNotNull()))
n_validas = df.count()
print(f"filas leídas: {n_raw:,} | con etiqueta y texto válidos: {n_validas:,} "
      f"| descartadas: {n_raw - n_validas:,} ({(n_raw - n_validas) / max(n_raw, 1):.1%})")

**Veracidad:** aunque leímos con `multiLine`, siempre habrá filas con `recommended` inválido (nulos, texto corrido). Esa cifra es parte del análisis, no un detalle: si fuera alta, habría que revisar el parseo antes de modelar.

Otra decisión de calidad: un modelo de texto sirve para **un** idioma. Vemos la mezcla y nos quedamos con inglés.

In [ ]:
if "language" in cols:
    df.groupBy("language").count().orderBy(F.desc("count")).show(5, truncate=False)
    df = df.filter(F.lower(F.col("language")) == "english")
else:
    print("No hay columna 'language': se asume un solo idioma")

## 2. Limpieza de texto

Una sola función `limpiar()` para entrenar **y** para puntuar texto nuevo (así no hay *training-serving skew*). Quita URLs y etiquetas BBCode de Steam (`[h1]`, `[b]`), y descarta reseñas de menos de 10 caracteres ("gg", "ok"), que no tienen señal.

También se eliminan duplicados por juego + texto: el spam ("10/10" × miles) infla el vocabulario y deja reseñas idénticas en train y test (una fuga silenciosa).

In [ ]:
def limpiar(d):
    return (d
            .withColumn("texto", F.regexp_replace(F.lower(F.col("review")), r"https?://\S+", " "))
            .withColumn("texto", F.regexp_replace("texto", r"\[/?[a-z0-9=*]+\]", " "))
            .withColumn("texto", F.regexp_replace("texto", r"\s+", " "))
            .filter(F.length("texto") >= 10))

df = limpiar(df)
df = df.dropDuplicates(["texto"] + (["app_id"] if "app_id" in cols else []))

# Variables estructuradas del autor para el modelo B. log1p: playtime tiene una cola larguísima.
num_cols = [c for c in ("author_playtime_forever", "author_num_games_owned", "author_num_reviews") if c in cols]
for c in num_cols:
    df = df.withColumn(c + "_log", F.log1p(F.coalesce(F.col(c).cast(DoubleType()), F.lit(0.0))))
num_feats = [c + "_log" for c in num_cols]

df = df.select("label", "texto", *num_feats).persist(StorageLevel.MEMORY_AND_DISK)
n = df.count()
print(f"reseñas utilizables (inglés, sin duplicados): {n:,}")

## 3. Desbalance de clases

Si ~85% de las reseñas recomienda el juego, un modelo que siempre responda "recomienda" ya "acierta" 85%: **accuracy no sirve**. Se compensa con un peso inverso a la frecuencia de cada clase (`weightCol`) y se evalúa con AUC-PR y con precisión/recall de la clase **negativa**, que es la que le importa a un estudio de videojuegos.

In [ ]:
balance = {r["label"]: r["count"] for r in df.groupBy("label").count().collect()}
n_pos, n_neg = balance.get(1.0, 0), balance.get(0.0, 0)
print(f"recomendadas: {n_pos:,} ({n_pos / n:.1%}) | NO recomendadas: {n_neg:,} ({n_neg / n:.1%})")

w_pos, w_neg = n / (2.0 * n_pos), n / (2.0 * n_neg)
df = df.withColumn("peso", F.when(F.col("label") == 1.0, w_pos).otherwise(w_neg))

# Split ANTES de ajustar nada: el vocabulario y el IDF solo se aprenden de train.
train, test = df.randomSplit([0.8, 0.2], seed=42)

## 4. Modelo A: solo texto (TF-IDF + regresión logística)

`CountVectorizer` (y no `HashingTF`) porque conserva el vocabulario: eso permite, más abajo, saber **qué palabra** es cada coeficiente. `minDF=50` descarta palabras que aparecen en menos de 50 reseñas (con una fracción chica, bájalo a 5).

In [ ]:
MIN_DF, VOCAB = 50, 20000

tokenizer = RegexTokenizer(inputCol="texto", outputCol="tokens", pattern="[^a-z']+", minTokenLength=2)
stop = StopWordsRemover(inputCol="tokens", outputCol="tokens_limpios")
cv = CountVectorizer(inputCol="tokens_limpios", outputCol="tf", vocabSize=VOCAB, minDF=MIN_DF)
idf = IDF(inputCol="tf", outputCol="tfidf")
lr_a = LogisticRegression(featuresCol="tfidf", labelCol="label", weightCol="peso",
                          regParam=0.01, elasticNetParam=0.0, maxIter=30)
pipeline_a = Pipeline(stages=[tokenizer, stop, cv, idf, lr_a])

In [ ]:
%%time
modelo_a = pipeline_a.fit(train)
pred_a = modelo_a.transform(test).persist(StorageLevel.MEMORY_AND_DISK)

In [ ]:
roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
pr  = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")

def reporte(nombre, pred):
    auc_pr = pr.evaluate(pred)
    print(f"[{nombre}] AUC-ROC = {roc.evaluate(pred):.4f} | AUC-PR = {auc_pr:.4f}")
    cm = {(r["label"], r["prediction"]): r["count"]
          for r in pred.groupBy("label", "prediction").count().collect()}
    tp0, fn0, fp0 = cm.get((0.0, 0.0), 0), cm.get((0.0, 1.0), 0), cm.get((1.0, 0.0), 0)
    print(f"  reseñas NEGATIVAS -> precisión {tp0 / max(tp0 + fp0, 1):.3f} | recall {tp0 / max(tp0 + fn0, 1):.3f}")
    pred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
    return auc_pr

auc_a = reporte("modelo A: solo texto", pred_a)

## 5. Interpretación: ¿qué palabras separan positivas de negativas?

Cada coeficiente de la regresión corresponde a una palabra del vocabulario. Negativo = empuja hacia "NO recomienda"; positivo = hacia "recomienda".

⚠️ Es **correlación, no causa**: "refund" aparece en reseñas negativas porque la gente lo menciona al quejarse, no porque la palabra provoque el sentimiento.

In [ ]:
vocab = modelo_a.stages[2].vocabulary
coefs = modelo_a.stages[-1].coefficients.toArray()
ranking = sorted(zip(vocab, coefs), key=lambda t: t[1])

print("→ NO recomienda:", ", ".join(f"{w} ({c:+.2f})" for w, c in ranking[:20]))
print()
print("→ recomienda:   ", ", ".join(f"{w} ({c:+.2f})" for w, c in ranking[-20:][::-1]))

## 6. Modelo B: texto + variables estructuradas del autor

¿Aportan algo el tiempo jugado, los juegos que posee y las reseñas que ha escrito, además del texto? Se mide, no se supone.

In [ ]:
%%time
if num_feats:
    assembler = VectorAssembler(inputCols=["tfidf"] + num_feats, outputCol="features")
    lr_b = LogisticRegression(featuresCol="features", labelCol="label", weightCol="peso",
                              regParam=0.01, elasticNetParam=0.0, maxIter=30)
    modelo_b = Pipeline(stages=[tokenizer, stop, cv, idf, assembler, lr_b]).fit(train)
    auc_b = reporte("modelo B: texto + estructurado", modelo_b.transform(test))
    delta = auc_b - auc_a
    print(f"AUC-PR B - A = {delta:+.4f} -> " +
          ("las variables del autor aportan señal que el texto no tiene" if delta > 0.01
           else "el texto ya contiene casi toda la señal: el pipeline extra no se justifica"))
else:
    print("No hay columnas author_*: se omite el modelo B")

## 7. Guardar el modelo y puntuar reseñas nuevas

El `PipelineModel` guarda el vocabulario, el IDF y los coeficientes juntos: es el objeto que se cargaría en un servicio de predicción.

In [ ]:
modelo_a.write().overwrite().save(RUTA_MODELO)

nuevas = spark.createDataFrame([
    ("Absolutely love this game, hundreds of hours and still fun. Best purchase this year!",),
    ("Constant crashes after the last patch. Unplayable, I want a refund.",),
    ("It's okay I guess, nothing special but not terrible either.",),
], ["review"])

(modelo_a.transform(limpiar(nuevas))
    .select("review", F.round(vector_to_array("probability")[1], 3).alias("p_recomienda"), "prediction")
    .show(truncate=60))

In [ ]:
df.unpersist()
pred_a.unpersist()

## Para practicar

1. **Escala:** sube `FRACCION` (0.05 → 0.2 → 1.0) y anota cómo cambian los tiempos de `fit`. ¿Qué parte crece más, leer/limpiar o entrenar?
2. **Por juego:** agrupa por `app_name` el % de reseñas que el modelo marca como negativas contra el % real. ¿Qué juegos rompen el modelo?
3. **Hiperparámetros:** envuelve `pipeline_a` en un `CrossValidator` con una rejilla sobre `regParam` y `vocabSize`.
4. **Otro algoritmo:** compara contra `NaiveBayes`.
5. **Temas:** aplica `LDA` sobre las reseñas negativas para descubrir de qué se quejan.